# Module 6.3 — Self-Query Retriever

Converts a **natural language query** into both a semantic search and a **structured metadata filter**.

Example: *'Show me sci-fi books by Isaac Asimov after 1950'* →
- Semantic: `sci-fi books`
- Filter: `author == 'Isaac Asimov' AND year > 1950`

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.retrievers.self_query.base import SelfQueryRetriever
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.schema import Document

docs = [
    Document(page_content="A robot must not injure a human being.", metadata={"author": "Isaac Asimov", "year": 1950, "genre": "sci-fi"}),
    Document(page_content="Space exploration and colonisation of Mars.", metadata={"author": "Ray Bradbury", "year": 1950, "genre": "sci-fi"}),
    Document(page_content="A dystopian society controlled by Big Brother.", metadata={"author": "George Orwell", "year": 1949, "genre": "dystopia"}),
    Document(page_content="AI achieves consciousness and questions its existence.", metadata={"author": "Isaac Asimov", "year": 1956, "genre": "sci-fi"}),
    Document(page_content="The Hobbit and the journey to the Lonely Mountain.", metadata={"author": "Tolkien", "year": 1937, "genre": "fantasy"}),
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vs         = Chroma.from_documents(docs, embeddings, collection_name="sqr_demo")
llm        = ChatOpenAI(model="gpt-4o-mini", temperature=0)

metadata_field_info = [
    AttributeInfo(name="author", description="Author of the book", type="string"),
    AttributeInfo(name="year",   description="Year the book was published", type="integer"),
    AttributeInfo(name="genre",  description="Genre of the book (sci-fi, fantasy, dystopia)", type="string"),
]

retriever = SelfQueryRetriever.from_llm(
    llm, vs, "Books and their themes", metadata_field_info, verbose=True
)

for query in [
    "Find sci-fi books by Isaac Asimov",
    "Books published before 1950",
    "Fantasy books",
]:
    results = retriever.invoke(query)
    print(f"\nQuery: '{query}'")
    for d in results:
        print(f"  • [{d.metadata}] {d.page_content[:60]}")
